# Libraries

In [1]:
import os
import re
import json
import unicodedata
from pathlib import Path
from typing import Optional
import fitz          # PyMuPDF: extrae texto de PDFs con capa de texto
import requests
from tqdm import tqdm

# Paths

In [4]:
BASE_DIR    = Path("../")
RAW_DIR     = BASE_DIR / "data" / "bronze"   
PROC_DIR    = BASE_DIR / "data" / "silver" 
META_DIR    = BASE_DIR / "data" / "metadata"

# Text Extraction

In [5]:
def extract_text_from_pdf(pdf_path: Path, paper_id: str) -> Optional[str]:
    """
    Extract clean text from a scientific PDF.
    
    Strategy:
    1. Open with PyMuPDF
    2. For each page, extract text blocks with coordinates
    3. Sort blocks to handle 2-column layout
    4. Concatenate using line breaks while preserving structure
    """
    try:
        doc = fitz.open(pdf_path)
        full_text_parts = []
        
        for page_num, page in enumerate(doc):
            # Extract blocks with coordinates: (x0, y0, x1, y1, text, ...)
            blocks = page.get_text("blocks")
            
            # Filter out empty or image blocks
            text_blocks = [
                b for b in blocks
                if b[6] == 0  # type 0 = text
                and b[4].strip()  # non-empty
            ]
            
            # Sort for 2-column layout:
            # Group by vertical band (every 20px) and then by x
            # This avoids mixing left and right columns
            sorted_blocks = sorted(
                text_blocks,
                key=lambda b: (round(b[1] / 20), b[0])
            )
            
            page_text = "\n".join(b[4].strip() for b in sorted_blocks)
            full_text_parts.append(page_text)
        
        doc.close()
        return "\n\n".join(full_text_parts)
    
    except Exception as e:
        print(f"  ✗ Error extracting {paper_id}: {e}")
        return None

# Cleanning Functions

In [10]:
def remove_page_headers_footers(text: str) -> str:
    """
    Remove repetitive headers/footers such as:
    - "arXiv:2004.04906v3  [cs.CL]  5 Dec 2020"
    - "Preprint. Under review."
    - Standalone page numbers on their own line
    """
    # Standalone page numbers
    text = re.sub(r"^\s*\d{1,3}\s*$", "", text, flags=re.MULTILINE)

    # arXiv headers
    text = re.sub(r"arXiv:\d+\.\d+v\d+.*?\n", "", text)

    # "Preprint" notices
    text = re.sub(
        r"Preprint\.\s+Under review[^\n]*\n",
        "",
        text,
        flags=re.IGNORECASE,
    )

    return text


def normalize_whitespace(text: str) -> str:
    """
    Normalize whitespace:
    - Collapse multiple empty lines into at most one empty line
    - Remove trailing spaces on each line
    - Preserve paragraph breaks (2+ empty lines -> 1 empty line)

    IMPORTANT: Do not remove paragraph breaks - they are natural boundaries
    for semantic chunking on Day 2.
    """
    # Clean trailing spaces on each line
    lines = [line.rstrip() for line in text.split("\n")]

    # Collapse consecutive empty lines
    cleaned = []
    prev_empty = False

    for line in lines:
        is_empty = len(line.strip()) == 0
        if is_empty and prev_empty:
            continue
        cleaned.append(line)
        prev_empty = is_empty

    return "\n".join(cleaned)


def fix_hyphenation(text: str) -> str:
    """
    Fix end-of-line hyphenation from PDF word wrapping:
    "trans-\nformer" -> "transformer"
    "atten-\ntion" -> "attention"

    Only apply when a hyphen appears at end of line and the word continues
    on the next line.
    """
    # Pattern: word-\nword
    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)
    return text


def remove_references_section(text: str) -> str:
    """
    Remove the References section at the end of the paper.

    Why? References are mostly noise for semantic retrieval:
    - "Vaswani, A., et al. (2017). Attention is all you need."
    - They usually do not add directly retrievable conceptual content
    - But they can consume ~10% of paper text

    Conservative strategy: remove only from "References" to the end.
    If no clear references section is found, keep everything.
    """
    # Match "References" / "Bibliography" as a standalone line
    # (re.MULTILINE makes ^ and $ match line boundaries, not just string ends)
    ref_patterns = [
        r"^References\s*$",
        r"^Bibliography\s*$",
        r"^\d+\s*References\s*$",
    ]

    for pattern in ref_patterns:
        match = re.search(pattern, text, re.MULTILINE | re.IGNORECASE)
        if match:
            # Only remove if it appears in the last 25% of the document
            # (avoids false positives in papers discussing references)
            if match.start() > len(text) * 0.75:
                return text[:match.start()].strip()

    return text


def clean_paper_text(raw_text: str) -> str:
    """
    Full cleaning pipeline. Applies functions in order.
    Order matters: fix_hyphenation before whitespace normalization.
    """
    text = raw_text
    text = remove_page_headers_footers(text)
    text = fix_hyphenation(text)
    text = remove_references_section(text)
    text = normalize_whitespace(text)

    # Unicode normalization: convert equivalent special characters
    # Example: 'ﬁ' ligature -> 'fi'
    text = unicodedata.normalize("NFKC", text)

    return text.strip()

# Clean and Process Papers

In [11]:
corpus_stats = []

# Make sure output directories exist
PROC_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# Discover all PDFs in Bronze folder
pdf_files = sorted(RAW_DIR.glob("*.pdf"))

if not pdf_files:
    print(f"✗ No PDF files found in {RAW_DIR}")
else:
    print(f"Found {len(pdf_files)} PDF files in {RAW_DIR}")

for pdf_path in pdf_files:
    paper_id = pdf_path.stem
    txt_path = PROC_DIR / f"{paper_id}.txt"

    # Extract raw text
    raw_text = extract_text_from_pdf(pdf_path, paper_id)
    if raw_text is None:
        continue

    # Clean
    clean_text = clean_paper_text(raw_text)
    if not clean_text.strip():
        print(f"  ✗ {paper_id}: Empty text after cleaning, skipping")
        continue

    # Save cleaned text
    txt_path.write_text(clean_text, encoding="utf-8")

    # Basic statistics
    char_count = len(clean_text)
    word_count = len(clean_text.split())
    para_count = len([p for p in clean_text.split("\n\n") if p.strip()])
    eq_count = clean_text.count("[EQ_START]")

    stats = {
        "id": paper_id,
        "pdf_file": pdf_path.name,
        "txt_file": txt_path.name,
        "chars": char_count,
        "words": word_count,
        "paragraphs": para_count,
        "equations": eq_count,
    }
    corpus_stats.append(stats)

    print(
        f"  ✓ {paper_id}: {word_count:,} words | "
        f"{para_count} paragraphs | {eq_count} equations"
    )

# Save corpus metadata
meta_path = META_DIR / "corpus_stats.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(corpus_stats, f, indent=2, ensure_ascii=False)

print(f"\n✓ Corpus processed. Stats saved to {meta_path}")
print(f"✓ Papers successfully processed: {len(corpus_stats)} / {len(pdf_files)}")

Found 10 PDF files in ..\data\bronze
  ✓ 1502.04681v3: 7,276 words | 26 paragraphs | 0 equations
  ✓ 1512.03385v1: 9,519 words | 45 paragraphs | 0 equations
  ✓ 1910.13461v1: 5,310 words | 16 paragraphs | 0 equations
  ✓ 2312.11514v3: 13,849 words | 126 paragraphs | 0 equations
  ✓ 2312.15872v1: 4,647 words | 60 paragraphs | 0 equations
  ✓ 2312.17482v2: 11,438 words | 73 paragraphs | 0 equations
  ✓ 2407.15516v1: 3,767 words | 11 paragraphs | 0 equations
  ✓ 2504.08386v1: 5,193 words | 26 paragraphs | 0 equations
  ✓ 2606.03748v1: 16,919 words | 168 paragraphs | 0 equations
  ✓ AttentionIsAllYouNeed: 4,633 words | 38 paragraphs | 0 equations

✓ Corpus processed. Stats saved to ..\data\metadata\corpus_stats.json
✓ Papers successfully processed: 10 / 10
